# Human-in-the-Loop — LangGraph

A simple workflow to see how LangGraph PAUSES a graph mid-run so a
HUMAN can review, approve, or edit the state before it continues —
built on top of the **Persistence** covered in `persistant.ipynb`
(a `checkpointer` is what makes the pause/resume possible at all).

**`interrupt()`:**
A function called INSIDE a node that PAUSES the graph right there,
surfaces a payload (whatever you pass it) back to whoever called
`invoke()`, and waits. It only works because the graph is compiled
with a `checkpointer` — the state up to that point is already saved,
so the pause can last indefinitely (even across process restarts).

**`Command(resume=...)`:**
How you RESUME a paused graph — passed as the input to `invoke()`
instead of a normal state dict. The value inside `resume=` becomes
the return value of the `interrupt()` call that paused the node, so
the node picks up exactly where it left off with the human's answer
in hand.

**What it covers:**

- A node that generates a short tweet, then calls `interrupt()` to
  pause for human APPROVAL before the tweet is considered final.
- Detecting a paused graph via `workflow.get_state(config).next` —
  a non-empty tuple means the graph is waiting on a human.
- Reading the pending interrupt payload off `get_state(config).tasks`.
- Resuming with `Command(resume="approve")` to accept the tweet
  as-is.
- Resuming with `Command(resume=<edited text>)` to have the human
  REWRITE the tweet instead of approving it.

In [ ]:
from typing import TypedDict


class TweetState(TypedDict):
    topic: str
    tweet: str
    approved: bool

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
def generate_tweet(state: TweetState) -> TweetState:
    prompt = f"Write a short, punchy tweet (under 280 characters) about: {state['topic']}"

    tweet = model.invoke(prompt).content

    return {"tweet": tweet}

In [ ]:
from langgraph.types import interrupt


def human_review(state: TweetState) -> TweetState:
    # pauses the graph here — the dict below is surfaced to whoever
    # called invoke()/resumed it, as the paused task's interrupt value
    decision = interrupt(
        {
            "question": "Approve this tweet, or send edited text to rewrite it.",
            "tweet": state["tweet"],
        }
    )

    if decision == "approve":
        return {"approved": True}

    # anything else is treated as the human's rewritten tweet
    return {"tweet": decision, "approved": True}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# checkpointer — required for interrupt()/resume to work at all
checkpointer = MemorySaver()

# define the graph
graph = StateGraph(TweetState)

# define nodes
graph.add_node("generate_tweet", generate_tweet)
graph.add_node("human_review", human_review)

# add edges
graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "human_review")
graph.add_edge("human_review", END)

# compile graph — pass the checkpointer here
workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
# print/visualize the graph
from IPython.display import Image, display

display(Image(workflow.get_graph().draw_mermaid_png()))

In [ ]:
# execute the graph — it runs generate_tweet, then PAUSES inside
# human_review and returns control back here instead of finishing
config = {"configurable": {"thread_id": "1"}}

initial_state = {"topic": "morning coffee"}

result = workflow.invoke(initial_state, config=config)

print(result)

In [ ]:
# the graph is PAUSED, not finished — `next` shows which node is
# waiting to run once we resume
print(workflow.get_state(config).next)

# the pending interrupt payload — this is what a UI would show a
# human reviewer
print(workflow.get_state(config).tasks[0].interrupts)

In [ ]:
from langgraph.types import Command

# resume — the human approves the tweet as-is, so "approve" becomes
# the return value of the interrupt() call inside human_review
final_state = workflow.invoke(Command(resume="approve"), config=config)

print(final_state)

## Resuming with an edit instead of an approval

Same graph, but this time the human rejects the draft and sends back
their own rewritten text instead of `"approve"`. Using a fresh
`thread_id` so this run doesn't collide with the one above.

In [ ]:
config2 = {"configurable": {"thread_id": "2"}}

initial_state = {"topic": "monday mornings"}

result = workflow.invoke(initial_state, config=config2)

print(result)

In [ ]:
# resume — instead of approving, the human sends back their own
# rewritten tweet text, which the node stores as the final `tweet`
final_state = workflow.invoke(
    Command(resume="Mondays aren't the problem. Your coffee is just too weak."),
    config=config2,
)

print(final_state)